# 📊 Notebook 01 — Data Acquisition & Exploration

**Multimodal RCA Engine — Phase 1: Log Extraction & Processing**

This notebook covers:
1. Automated download of LogHub datasets (HDFS, BGL, OpenStack)
2. Initial data exploration and statistics
3. Log format comparison across datasets
4. Exploratory Data Analysis (EDA) with visualizations

---

## 1.1 — Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datetime import datetime

from src.utils import (
    ensure_directories,
    load_config,
    download_and_extract_dataset,
    read_log_file,
    get_file_size_mb,
    count_lines,
    print_section,
    RAW_DIR,
    FIGURES_DIR,
)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

# Create directories
ensure_directories()

print("✅ Setup complete!")
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📁 Raw data dir: {RAW_DIR}")

## 1.2 — Load Dataset Configuration

In [ ]:
config = load_config()

print_section("LogHub Datasets")
for key, info in config['loghub']['datasets'].items():
    print(f"  📦 {info['name']}")
    print(f"     Description: {info['description']}")
    print(f"     Size: {info['size']}")
    print(f"     Labels: {'✅' if info['labels'] else '❌'}")
    print(f"     Priority: {info['priority']}")
    print()

## 1.3 — Download Datasets

We download from [Zenodo](https://zenodo.org/records/8196385) (hosted by LogHub).

> ⚠️ **Note**: HDFS_v1 is ~1.5GB, BGL is ~700MB. Download may take several minutes.

In [ ]:
# Download HDFS_v1 (Primary dataset — has anomaly labels)
print_section("Downloading HDFS_v1")
hdfs_dir = download_and_extract_dataset('hdfs_v1', config)

In [ ]:
# Download BGL (Secondary dataset)
print_section("Downloading BGL")
bgl_dir = download_and_extract_dataset('bgl', config)

In [ ]:
# Download OpenStack (Tertiary — small, cloud-related)
print_section("Downloading OpenStack")
openstack_dir = download_and_extract_dataset('openstack', config)

## 1.4 — Explore Downloaded Files

In [ ]:
def explore_directory(dir_path, name):
    """List all files in a dataset directory with sizes."""
    dir_path = Path(dir_path)
    print_section(f"Files in {name}")
    
    if not dir_path.exists():
        print(f"  ❌ Directory does not exist: {dir_path}")
        return []
    
    files = []
    for f in sorted(dir_path.rglob('*')):
        if f.is_file() and not f.name.endswith(('.zip', '.tar.gz', '.tgz')):
            size_mb = get_file_size_mb(f)
            rel_path = f.relative_to(dir_path)
            files.append({'file': str(rel_path), 'size_mb': size_mb})
            print(f"  📄 {rel_path:<40s} {size_mb:>10.2f} MB")
    
    return files

hdfs_files = explore_directory(hdfs_dir, "HDFS_v1")
bgl_files = explore_directory(bgl_dir, "BGL")
openstack_files = explore_directory(openstack_dir, "OpenStack")

## 1.5 — Preview Raw Log Samples

Let's look at the first few lines of each log file to understand their format.

In [ ]:
def preview_logs(dir_path, file_pattern='*.log', n_lines=10, name=""):
    """Preview the first N lines of log files."""
    dir_path = Path(dir_path)
    log_files = list(dir_path.rglob(file_pattern))
    
    if not log_files:
        # Try other extensions
        for ext in ['*.log', '*.csv', '*.txt', '*']:
            log_files = [f for f in dir_path.rglob(ext) if f.is_file() and f.suffix not in ['.zip', '.gz', '.md']]
            if log_files:
                break
    
    for lf in log_files[:3]:  # Show max 3 files
        print_section(f"{name} — {lf.name}")
        lines = read_log_file(lf, max_lines=n_lines)
        for i, line in enumerate(lines):
            print(f"  [{i+1:3d}] {line[:150]}")
        
        total = count_lines(lf)
        size = get_file_size_mb(lf)
        print(f"\n  📊 Total lines: {total:,} | Size: {size:.2f} MB")

preview_logs(hdfs_dir, name="HDFS")

In [ ]:
preview_logs(bgl_dir, name="BGL")

In [ ]:
preview_logs(openstack_dir, name="OpenStack")

## 1.6 — Basic Statistics & Format Comparison

In [ ]:
import re

def quick_log_stats(dir_path, name, sample_size=50000):
    """Compute quick statistics from a sample of logs."""
    dir_path = Path(dir_path)
    
    # Find the main log file
    log_files = [f for f in dir_path.rglob('*') if f.is_file() 
                 and f.suffix in ['.log', '.csv', '.txt', '']
                 and f.stat().st_size > 1000]
    
    if not log_files:
        print(f"  ❌ No log files found in {dir_path}")
        return None
    
    # Use the largest file
    main_file = max(log_files, key=lambda f: f.stat().st_size)
    
    lines = read_log_file(main_file, max_lines=sample_size)
    
    # Count log levels
    level_pattern = re.compile(r'\b(INFO|WARN|WARNING|ERROR|FATAL|DEBUG|TRACE|CRITICAL)\b', re.IGNORECASE)
    levels = Counter()
    for line in lines:
        match = level_pattern.search(line)
        if match:
            levels[match.group(1).upper()] += 1
    
    # Average line length
    avg_len = np.mean([len(l) for l in lines]) if lines else 0
    
    stats = {
        'dataset': name,
        'main_file': main_file.name,
        'sample_size': len(lines),
        'total_lines': count_lines(main_file),
        'file_size_mb': get_file_size_mb(main_file),
        'avg_line_length': avg_len,
        'log_levels': dict(levels),
    }
    
    return stats

print_section("Computing Statistics...")
stats_hdfs = quick_log_stats(hdfs_dir, "HDFS")
stats_bgl = quick_log_stats(bgl_dir, "BGL")
stats_openstack = quick_log_stats(openstack_dir, "OpenStack")

all_stats = [s for s in [stats_hdfs, stats_bgl, stats_openstack] if s is not None]

# Display comparison table
comparison = pd.DataFrame([{
    'Dataset': s['dataset'],
    'Main File': s['main_file'],
    'Total Lines': f"{s['total_lines']:,}",
    'File Size (MB)': f"{s['file_size_mb']:.1f}",
    'Avg Line Length': f"{s['avg_line_length']:.0f}",
    'Log Levels Found': ', '.join(f"{k}:{v}" for k, v in sorted(s['log_levels'].items(), key=lambda x: -x[1])[:5]),
} for s in all_stats])

comparison

## 1.7 — EDA Visualizations

In [ ]:
# === Log Level Distribution (Pie Charts) ===

fig, axes = plt.subplots(1, len(all_stats), figsize=(6 * len(all_stats), 6))
if len(all_stats) == 1:
    axes = [axes]

colors = {
    'INFO': '#2196F3',
    'WARN': '#FF9800', 'WARNING': '#FF9800',
    'ERROR': '#F44336',
    'FATAL': '#9C27B0',
    'DEBUG': '#4CAF50',
    'TRACE': '#607D8B',
    'CRITICAL': '#E91E63',
}

for ax, s in zip(axes, all_stats):
    levels = s['log_levels']
    if levels:
        labels = list(levels.keys())
        values = list(levels.values())
        pie_colors = [colors.get(l, '#9E9E9E') for l in labels]
        
        wedges, texts, autotexts = ax.pie(
            values, labels=labels, colors=pie_colors,
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11}
        )
        ax.set_title(f"{s['dataset']} — Log Level Distribution", fontsize=14, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'No levels found', ha='center', va='center')
        ax.set_title(f"{s['dataset']}")

plt.suptitle('Log Level Distribution Across Datasets', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '01_log_level_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: results/figures/01_log_level_distribution.png")

In [ ]:
# === Dataset Size Comparison (Bar Chart) ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

dataset_names = [s['dataset'] for s in all_stats]
total_lines = [s['total_lines'] for s in all_stats]
file_sizes = [s['file_size_mb'] for s in all_stats]

bar_colors = ['#1976D2', '#388E3C', '#F57C00']

# Total lines
bars1 = ax1.bar(dataset_names, total_lines, color=bar_colors[:len(all_stats)], edgecolor='white', linewidth=1.5)
ax1.set_ylabel('Number of Lines', fontsize=12)
ax1.set_title('Total Log Lines', fontsize=14, fontweight='bold')
for bar, val in zip(bars1, total_lines):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(total_lines)*0.01,
             f'{val:,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# File sizes
bars2 = ax2.bar(dataset_names, file_sizes, color=bar_colors[:len(all_stats)], edgecolor='white', linewidth=1.5)
ax2.set_ylabel('Size (MB)', fontsize=12)
ax2.set_title('File Size', fontsize=14, fontweight='bold')
for bar, val in zip(bars2, file_sizes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(file_sizes)*0.01,
             f'{val:.1f} MB', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Dataset Size Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '01_dataset_size_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Line Length Distribution (Histograms) ===

fig, axes = plt.subplots(1, len(all_stats), figsize=(6 * len(all_stats), 5))
if len(all_stats) == 1:
    axes = [axes]

for ax, s in zip(axes, all_stats):
    dir_map = {'HDFS': hdfs_dir, 'BGL': bgl_dir, 'OpenStack': openstack_dir}
    d = dir_map.get(s['dataset'])
    if d:
        log_files = [f for f in Path(d).rglob('*') if f.is_file() and f.stat().st_size > 1000
                     and f.suffix in ['.log', '.csv', '.txt', '']]
        if log_files:
            main_file = max(log_files, key=lambda f: f.stat().st_size)
            sample = read_log_file(main_file, max_lines=10000)
            lengths = [len(l) for l in sample]
            
            ax.hist(lengths, bins=50, color=bar_colors[all_stats.index(s) % len(bar_colors)], 
                    alpha=0.8, edgecolor='white')
            ax.axvline(np.mean(lengths), color='red', linestyle='--', label=f'Mean: {np.mean(lengths):.0f}')
            ax.set_xlabel('Line Length (characters)')
            ax.set_ylabel('Frequency')
            ax.set_title(f"{s['dataset']} — Log Line Length", fontweight='bold')
            ax.legend()

plt.suptitle('Log Line Length Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '01_line_length_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 1.8 — HDFS Anomaly Label Analysis

HDFS_v1 comes with `anomaly_label.csv` — let's explore the label distribution.

In [ ]:
# Find and load anomaly labels
label_files = list(Path(hdfs_dir).rglob('*anomaly*label*'))
if not label_files:
    label_files = list(Path(hdfs_dir).rglob('*.csv'))

print("Label files found:")
for f in label_files:
    print(f"  📄 {f.name} ({get_file_size_mb(f):.2f} MB)")

if label_files:
    label_df = pd.read_csv(label_files[0])
    print(f"\n📊 Label file shape: {label_df.shape}")
    print(f"\nColumns: {list(label_df.columns)}")
    display(label_df.head(10))
    
    # Distribution
    print("\n📊 Label Distribution:")
    print(label_df.iloc[:, -1].value_counts())

In [ ]:
# === Anomaly Distribution Pie Chart ===
if label_files:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    label_col = label_df.columns[-1]
    counts = label_df[label_col].value_counts()
    
    # Pie chart
    pie_colors_anomaly = ['#4CAF50', '#F44336']
    ax1.pie(counts.values, labels=counts.index, colors=pie_colors_anomaly,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 13})
    ax1.set_title('HDFS Anomaly Label Distribution', fontsize=14, fontweight='bold')
    
    # Bar chart
    bars = ax2.bar(counts.index.astype(str), counts.values, color=pie_colors_anomaly, 
                   edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, counts.values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.01,
                 f'{val:,}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Count', fontsize=12)
    ax2.set_title('Normal vs Anomaly Count', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / '01_hdfs_anomaly_distribution.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("💾 Saved: results/figures/01_hdfs_anomaly_distribution.png")

## 1.9 — Summary & Next Steps

### ✅ What we accomplished:
- Downloaded 3 log datasets (HDFS_v1, BGL, OpenStack)
- Explored file structures and log formats
- Analyzed log level distributions
- Examined HDFS anomaly labels

### 📌 Key Findings:
| Aspect | HDFS | BGL | OpenStack |
|--------|------|-----|----------|
| Labels | ✅ block-level anomaly | ✅ alert labels | ✅ available |
| Use Case | Distributed file system | Supercomputer | Cloud platform |
| Primary Focus | Session-based anomaly | Event-based anomaly | Service failures |

### ➡️ Next: Notebook 02 — Log Parsing
We'll parse these raw logs into structured data using regex patterns and the Drain algorithm.